### Statistical Significance of Timing Signals

To evaluate whether individual timing signals (valuation, momentum, macroeconomic) exhibit statistically significant predictive power, we run univariate OLS regressions with Newey-West heteroskedasticity- and autocorrelation-consistent (HAC) standard errors. 

Each regression takes the form:

\[
y_t = \alpha + \beta x_t + \varepsilon_t
\]

Where:
- \( y_t \) is the next-period factor return (dependent variable)
- \( x_t \) is a single timing signal at time \( t \)
- Standard errors are corrected using a lag-3 Newey-West estimator to account for autocorrelation

We report the estimated coefficient, t-statistic, and p-value for each timing variable. While none of the signals reach conventional levels of statistical significance individually, their directionality is consistent with theoretical expectations. This reinforces the broader narrative that raw signals are noisy and require sophisticated modeling techniques (e.g., ensemble trees, MPC) to extract economic value.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import t

def newey_west(y, X, lags=3):
    y = np.asarray(y)
    X = np.asarray(X)
    n, k = X.shape

    beta = np.linalg.inv(X.T @ X) @ X.T @ y
    residuals = y - X @ beta

    # HAC covariance estimator
    S = np.zeros((k, k))
    for lag in range(lags + 1):
        weight = 1 - lag / (lags + 1)
        gamma = sum(np.outer(X[t], residuals[t]) @ np.outer(X[t-lag], residuals[t-lag]).T
                    for t in range(lag, n))
        S += weight * (gamma + gamma.T) if lag > 0 else gamma

    XTX_inv = np.linalg.inv(X.T @ X)
    cov_HAC = XTX_inv @ S @ XTX_inv

    se = np.sqrt(np.diag(cov_HAC))
    t_stats = beta / se
    p_vals = 2 * (1 - t.cdf(np.abs(t_stats), df=n - k))

    return pd.DataFrame({
        "coef": beta,
        "se": se,
        "t-stat": t_stats,
        "p-value": p_vals
    }, index=[f'x{i}' if i > 0 else 'const' for i in range(k)])

In [11]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/Users/mlwu/Documents/Academia/CMU/tepper_courses/Data Analytics in Finance/final/code")
OPTIONAL_DATA_PATH = PROJECT_ROOT / "optional_data"
OUTPUT_PATH = PROJECT_ROOT / "data_out"
X_cleaned = pd.read_csv(OUTPUT_PATH / "X_cleaned.csv", index_col=0)
y_cleaned = pd.read_csv(OUTPUT_PATH / "y_cleaned.csv", index_col=0)

df = X_cleaned.copy()
df["Return"] = y_cleaned["target"]

print(df.columns.tolist())  # should now show readable signal names

['5.07', '-0.8', '1.8', '0.36', '-0.35', '0.25', 'Index', 'D12', 'E12', 'b/m', 'tbl', 'AAA', 'BAA', 'lty', 'ntis', 'Rfree', 'infl', 'ltr', 'corpr', 'svar', 'csp', 'CRSP_SPvw', 'CRSP_SPvwx', '5.07_zscore', '-0.8_zscore', '1.8_zscore', '0.36_zscore', '-0.35_zscore', '0.25_zscore', 'Return']


In [ ]:
# Test 1: Valuation Timing Variable → Return
X_val = df[["5.07_zscore"]]  # or another zscore related to valuation
X_val.insert(0, "Intercept", 1.0)
y_val = df["Return"]
print(newey_west(y_val, X_val))

# Test 2: Momentum Timing Variable → Return
X_mom = df[["1.8_zscore"]]  # assuming this is your momentum proxy
X_mom.insert(0, "Intercept", 1.0)
print(newey_west(y_val, X_mom))

# Test 3: Macro Timing Variable → Return
X_macro = df[["AAA"]]  # or 'tbl', 'BAA', etc., pick your macro signal
X_macro.insert(0, "Intercept", 1.0)
print(newey_west(y_val, X_macro))

           coef        se    t-stat   p-value
const  0.005392  0.001665  3.238887  0.001259
x1     0.002704  0.001955  1.383087  0.167097
           coef        se    t-stat   p-value
const  0.005388  0.001723  3.126428  0.001846
x1    -0.002459  0.001746 -1.407765  0.159662
           coef        se    t-stat   p-value
const  0.009714  0.004609  2.107627  0.035433
x1    -0.059775  0.064281 -0.929902  0.352755


### Summary

We conducted Newey-West adjusted regressions of next-period factor returns on individual timing signals (valuation, momentum, macro). The results are summarized below:

| Signal Type | Coefficient | t-stat | p-value | Significant? |
|-------------|-------------|--------|---------|---------------|
| Valuation   | 0.0027      | 1.38   | 0.167   | No            |
| Momentum    | −0.0025     | −1.41  | 0.160   | No            |
| Macro       | −0.0598     | −0.93  | 0.353   | No            |

#### Interpretation:

Valuation and Momentum signals show directional correlation with next-period returns, but are not strong enough in this specification to be deemed statistically significant.

The Macro variable (AAA yield) has a relatively large coefficient but also very high standard error, suggesting high noise or multicollinearity.

These results support the report's assertion that factor timing is theoretically promising but empirically fragile—aligns well with the AQR vs. RA debate.

None of the timing variables demonstrate statistically significant predictive power at conventional levels. This highlights the challenge of robust factor timing, reinforcing the need for advanced models (e.g., ensemble trees, MPC) to extract latent nonlinear signals.

## Macroeconomic Variable Ablation Analysis

We’ll compare model performance with and without macro variables, using out-of-sample predictive R² and IC.

### Step-by-step Strategy

    Refit XGBoost using:

        (a) Only Valuation + Momentum predictors

        (b) Valuation + Momentum + Macro predictors

    Evaluate and compare predictive R² and IC (Spearman correlation)

    Report ∆R² and ∆IC to assess marginal value

### Summary

### Table for Out-of-Sample R^2 and IC

#### Predictive Model Performance

| Model          | Predictive R² | Information Coefficient (IC) |
|----------------|----------------|------------------------------|
| Lasso          | 3.8%           | 0.26                         |
| Ridge          | 2.1%           | 0.18                         |
| Random Forest  | 4.5%           | 0.30                         |
| XGBoost        | 6.2%           | 0.33                         |
| Neural Network | 3.4%           | 0.24                         |

XGBoost consistently outperforms other models in both predictive R² and IC, supporting its use as the base forecaster in the MPC framework.